In [6]:
import requests
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
load_dotenv()

# 1. API 데이터 가져오기 (가공 전)
def fetch_youth_policy_data(page_num=1):
    url = 'https://www.youthcenter.go.kr/go/ythip/getPlcy'
    params = {
        'apiKeyNm': '967a7a21-2313-4206-8427-ef541c73b12e',
        'pageNum': page_num,
        'pageSize': 200, # 한 번에 많이 가져오기
        'rtnType': 'json'
    }
    response = requests.get(url, params=params)
    return response.json()['result']['youthPolicyList']

# 2. JSON -> Document 객체 변환 (제일 중요한 과정!)
raw_data = fetch_youth_policy_data()
docs = []

for item in raw_data:
    # page_content: LLM이 문맥을 이해할 수 있도록 중요한 정보들을 합쳐서 텍스트로 만듦
    content = f"""
    정책명: {item.get('plcyNm', '')}
    정책설명: {item.get('plcyExplnCn', '')}
    지원내용: {item.get('plcySprtCn', '')}
    신청방법: {item.get('plcyAplyMthdCn', '')}
    """

    # metadata: 나중에 필터링(SQL처럼) 조건으로 쓸 값들
    metadata = {
        "plcyNo": item.get('plcyNo', ''),
        "region": item.get('zipCd', ''), # 전국 확대 시 이 필터 활용
        "age_min": item.get('sprtTrgtMinAge', ''),
        "age_max": item.get('sprtTrgtMaxAge', ''),
        "category": item.get('lclsfNm', '')
    }

    docs.append(Document(page_content=content, metadata=metadata))

# 3. ChromaDB에 저장 (영구 저장)
print(f"총 {len(docs)}개의 문서를 벡터 DB로 변환합니다...")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(), # OpenAI 키가 환경변수에 설정되어 있어야 함
    persist_directory="./data/json_data_db"
)

print("DB 저장 완료!")
# 1. DB 불러오기
vectordb = Chroma(
    persist_directory="./data/json_data_db",
    embedding_function=OpenAIEmbeddings()
)

# 2. 테스트용 질문 리스트
test_queries = [
    "청년 월세 지원해주는 정책 있어?",
    "경기도에 사는 청년 정책 알려줘"
]

# 3. 검색 및 결과 확인
for query in test_queries:
    print(f"\n[질문]: {query}")
    # k=3은 가장 관련성 높은 문서 3개를 가져오라는 뜻
    results = vectordb.similarity_search_with_score(query, k=3)

    for i, (doc, score) in enumerate(results):
        # score는 거리를 의미합니다 (보통 낮을수록 유사도가 높음)
        print(f" --- 결과 {i+1} (점수: {score:.4f}) ---")
        print(f" 내용: {doc.page_content[:50]}...") # 내용 50자만 출력
        print(f" 메타데이터: {doc.metadata}")

총 200개의 문서를 벡터 DB로 변환합니다...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/var/folders/c4/qpkcr7fd3796r5m4nq3nvylr0000gn/T/ipykernel_57705/245755649.py:55: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


DB 저장 완료!

[질문]: 청년 월세 지원해주는 정책 있어?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


 --- 결과 1 (점수: 0.1926) ---
 내용: 
    정책명: 청년 월세 지원
    정책설명: 경제적 어려움을 겪고 있는 저소득 청년...
 메타데이터: {'age_max': '34', 'age_min': '19', 'category': '주거', 'plcyNo': '20260409005400212654', 'region': '31110,31140,31170,31200,31710'}
 --- 결과 2 (점수: 0.2120) ---
 내용: 
    정책명: 제주 청년 희망충전 월세 지원
    정책설명: 청년층의 주거비 부담경감...
 메타데이터: {'age_max': '39', 'age_min': '35', 'category': '주거', 'plcyNo': '20260422005400212846', 'region': '50110,50130'}
 --- 결과 3 (점수: 0.2193) ---
 내용: 
    정책명: 청년월세 지원
    정책설명: 청년층의 주거비 부담경감을 위해 부모와 ...
 메타데이터: {'age_max': '34', 'age_min': '19', 'category': '주거,주거', 'plcyNo': '20260421005400212801', 'region': '50110,50130'}

[질문]: 경기도에 사는 청년 정책 알려줘
 --- 결과 1 (점수: 0.2989) ---
 내용: 
    정책명: 청년정책 제안 경연대회
    정책설명: 청년정책 수요 당사자인 청년들이...
 메타데이터: {'age_max': '39', 'age_min': '19', 'category': '참여･기반', 'plcyNo': '20260414005400212739', 'region': '31110,31140,31170,31200,31710'}
 --- 결과 2 (점수: 0.3052) ---
 내용: 
    정책명: 서귀포시 청년정책협의체 운영
    정책설명: 청년들의 시정 참여 확대 ...
 메타데이터: {'age_max': '39